In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import re

# ======================
# 1. 크롬 드라이버 실행
# ======================
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

wait = WebDriverWait(driver, 10)

url = "https://www.vigloo.com/ko?tab=ranking"
driver.get(url)
time.sleep(3)


# ======================
# 2. 유틸 함수
# ======================
def clean_text_list(text):
    return [x.strip() for x in text.split("\n") if x.strip()]


def scroll_down(times=5):
    for _ in range(times):
        driver.execute_script("window.scrollBy(0, 700);")
        time.sleep(0.8)


def get_total_episode(body_text):
    """
    예:
    1-49
    50-76(완결)
    → 76 추출
    """
    ranges = re.findall(r"\d+\s*-\s*\d+(?:\(완결\))?", body_text)

    if ranges:
        nums = re.findall(r"\d+", " ".join(ranges))
        return max(map(int, nums))

    # fallback: 숫자 버튼에서 최대값
    nums = re.findall(r"\n(\d{1,3})\n", body_text)
    if nums:
        return max(map(int, nums))

    return ""


# ======================
# 3. 랭킹 카드 로딩
# ======================
scroll_down(5)

cards = driver.find_elements(
    By.XPATH,
    "//div[contains(@class,'bg-elevation-100') and .//img]"
)

print("카드 개수:", len(cards))


# ======================
# 4. 랭킹 카드 기본정보 수집
# ======================
ranking_items = []

for idx, card in enumerate(cards, start=1):
    try:
        text = card.text.strip()
        lines = clean_text_list(text)

        # 예시 lines:
        # ['1', '천재 아기의 인생 역전', '참교육', '비밀신분', '타임루프', '배신', 'HOT']

        rank = ""
        title = ""
        tags = []

        for line in lines:
            if line.isdigit():
                rank = line
                break

        # 숫자 / HOT / NEW 제외하고 첫 번째가 제목
        candidates = [
            line for line in lines
            if line not in [rank, "HOT", "NEW"]
        ]

        if candidates:
            title = candidates[0]
            tags = candidates[1:]

        if title:
            ranking_items.append({
                "rank": rank,
                "title": title,
                "tags": ", ".join(tags),
                "card_index": idx
            })

    except Exception as e:
        print("카드 기본정보 수집 오류:", e)

print("수집된 랭킹:", len(ranking_items))
print(ranking_items[:5])


# ======================
# 5. 카드 클릭해서 상세정보 수집
# ======================
results = []

for item in ranking_items:
    print(f"{item['rank']}위 {item['title']} 상세 수집 중...")

    # 랭킹 페이지로 다시 이동
    driver.get(url)
    time.sleep(2)
    scroll_down(max(1, item["card_index"] // 4))

    # 카드 다시 잡기
    cards = driver.find_elements(
        By.XPATH,
        "//div[contains(@class,'bg-elevation-100') and .//img]"
    )

    try:
        target_card = cards[item["card_index"] - 1]

        # 이미지 또는 카드 클릭
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", target_card)
        time.sleep(0.5)
        driver.execute_script("arguments[0].click();", target_card)

        time.sleep(3)

    except Exception as e:
        print("카드 클릭 실패:", item["title"], e)
        continue

    detail_url = driver.current_url

    # ======================
    # 상세 페이지 텍스트 수집
    # ======================
    body_text = driver.find_element(By.TAG_NAME, "body").text
    lines = clean_text_list(body_text)

    # 줄거리: 긴 문장 중 가장 긴 것
    synopsis_candidates = [
        line for line in lines
        if len(line) >= 40
        and "멤버십 구독" not in line
        and "무제한 시청하기" not in line
        and "비슷한 콘텐츠" not in line
        and "에피소드" not in line
    ]

    synopsis = max(synopsis_candidates, key=len) if synopsis_candidates else ""

    # 상세 장르: 예시 '환생 · 반전 · 사이다'
    detail_genre = ""
    for line in lines:
        if "·" in line and len(line) <= 40:
            detail_genre = line
            break

    # 총 회차 수
    total_episode = get_total_episode(body_text)

    results.append({
        "랭킹": item["rank"],
        "제목": item["title"],
        "랭킹_태그": item["tags"],
        "상세_장르": detail_genre,
        "줄거리": synopsis,
        "총회차": total_episode,
        "URL": detail_url
    })

    time.sleep(1)


# ======================
# 6. 저장
# ======================
df = pd.DataFrame(results)

df.to_csv("vigloo_ranking_result.csv", index=False, encoding="utf-8-sig")
df.to_excel("vigloo_ranking_result.xlsx", index=False)

print("완료!")
print(df)

driver.quit()

카드 개수: 10
수집된 랭킹: 10
[{'rank': '', 'title': '천재 아기의 인생 역전', 'tags': '참교육, 비밀신분, 타임루프, 배신', 'card_index': 1}, {'rank': '', 'title': '수상한 남편의 이중생활', 'tags': '로맨틱한, 원나잇, 재벌, 언더커버', 'card_index': 2}, {'rank': '', 'title': '닿을 수 있는 그녀', 'tags': '초고속결혼, 재벌, 로맨틱한', 'card_index': 3}, {'rank': '', 'title': '뚱보 아내는 조폭 마누라', 'tags': '불륜, 복수, 사이다, 빙의, 정치', 'card_index': 4}, {'rank': '', 'title': '비서할래 와이프할래?', 'tags': '선결혼후연애, 오피스, 로코, 재벌', 'card_index': 5}]
위 천재 아기의 인생 역전 상세 수집 중...
위 수상한 남편의 이중생활 상세 수집 중...
위 닿을 수 있는 그녀 상세 수집 중...
위 뚱보 아내는 조폭 마누라 상세 수집 중...
위 비서할래 와이프할래? 상세 수집 중...
위 국가권력급 상속자가 돌아왔다 상세 수집 중...
위 모쏠지옥 상세 수집 중...
위 타임리프 로맨스: 은밀한 복수 상세 수집 중...
위 장녀 결혼 일기 상세 수집 중...
위 오늘 한류스타와 이혼하겠습니다 상세 수집 중...
완료!
  랭킹                제목                 랭킹_태그 상세_장르  \
0         천재 아기의 인생 역전   참교육, 비밀신분, 타임루프, 배신   환생·   
1         수상한 남편의 이중생활   로맨틱한, 원나잇, 재벌, 언더커버  로맨스·   
2           닿을 수 있는 그녀       초고속결혼, 재벌, 로맨틱한  로맨스·   
3        뚱보 아내는 조폭 마누라   불륜, 복수, 사이다, 빙의, 정치   환생·   
4          비서할래 와이

In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

driver.get("https://www.vigloo.com/ko")
time.sleep(3)
scroll_down(3)

cards = driver.find_elements(By.XPATH, "//div[contains(@class,'vds-bundle-card')]")
print("카드 수:", len(cards))

for card in cards[:5]:
    print("-----")
    print(card.text)

카드 수: 39
-----
타임리프 로맨스: 은밀한 복수
시간여행
-----
국가권력급 상속자가 돌아왔다
참교육
-----
천재 아기의 인생 역전
참교육
-----
닿을 수 있는 그녀
초고속결혼
-----
수상한 남편의 이중생활
로맨틱한


In [15]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import re

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)


# 공통 함수
def clean_text_list(text):
    return [x.strip() for x in text.split("\n") if x.strip()]


def scroll_down(times=5):
    for _ in range(times):
        driver.execute_script("window.scrollBy(0, 700);")
        time.sleep(0.8)


def get_total_episode(body_text):
    ranges = re.findall(r"\d+\s*-\s*\d+(?:\(완결\))?", body_text)

    if ranges:
        nums = re.findall(r"\d+", " ".join(ranges))
        return max(map(int, nums))

    nums = re.findall(r"\n(\d{1,3})\n", body_text)
    if nums:
        return max(map(int, nums))

    return ""


def get_synopsis_from_detail():
    ps = driver.find_elements(By.CSS_SELECTOR, "p.text-gray-white")
    candidates = []

    for p in ps:
        txt = p.text.strip()
        if (
            len(txt) >= 30
            and "멤버십 구독" not in txt
            and "무제한 시청하기" not in txt
            and "비슷한 콘텐츠" not in txt
            and "에피소드" not in txt
            and "주소:" not in txt
            and "VIGLOO" not in txt
        ):
            candidates.append(txt)

    return max(candidates, key=len) if candidates else ""


def get_genre_from_detail():
    spans = driver.find_elements(By.CSS_SELECTOR, "span.text-gray-400")
    genre_list = []

    for span in spans:
        txt = span.text.strip()
        if txt and len(txt) <= 10 and txt not in genre_list:
            genre_list.append(txt)

    return " · ".join(genre_list)


# 인기 수집
popular_results = []

popular_url = "https://www.vigloo.com/ko"
print("\n===== 인기 수집 시작 =====")

driver.get(popular_url)
time.sleep(3)
scroll_down(5)

popular_cards = driver.find_elements(
    By.XPATH,
    "//div[contains(@class,'vds-bundle-card')]"
)

print("인기 카드 개수:", len(popular_cards))

popular_items = []

for idx, card in enumerate(popular_cards, start=1):
    lines = clean_text_list(card.text)

    if not lines:
        continue

    title = lines[0]
    desc = lines[1] if len(lines) >= 2 else ""

    popular_items.append({
        "순번": idx,
        "제목": title,
        "태그/설명": desc,
        "card_index": idx
    })

print("인기 수집 대상:", len(popular_items))


for item in popular_items:
    print(f"인기 / {item['순번']} / {item['제목']}")

    driver.get(popular_url)
    time.sleep(2)

    # 제목이 보일 때까지 스크롤하며 카드 찾기
    target = None

    for _ in range(12):
        xpath = f"//div[contains(@class,'vds-bundle-card')][.//h3[contains(normalize-space(), \"{item['제목']}\")]]"
        found = driver.find_elements(By.XPATH, xpath)

        if found:
            target = found[0]
            break

        driver.execute_script("window.scrollBy(0, 500);")
        time.sleep(0.7)

    if target is None:
        print("인기 카드 못 찾음:", item["제목"])
        continue

    try:
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", target)
        time.sleep(0.8)

        old_url = driver.current_url

        img = target.find_element(By.CSS_SELECTOR, "img[alt='thumbnail image']")
        driver.execute_script("arguments[0].click();", img)
        time.sleep(3)

        detail_url = driver.current_url

        if detail_url == old_url or "/video/" not in detail_url:
            print("인기 상세 이동 실패:", item["제목"])
            continue

    except Exception as e:
        print("인기 클릭 실패:", item["제목"], e)
        continue

    body_text = driver.find_element(By.TAG_NAME, "body").text
    lines = clean_text_list(body_text)

    genre = get_genre_from_detail()
    synopsis = get_synopsis_from_detail()
    total_episode = get_total_episode(body_text)

    popular_results.append({
        "차트": "인기",
        "순번": item["순번"],
        "제목": item["제목"],
        "태그/설명": item["태그/설명"],
        "상세_장르": genre,
        "줄거리": synopsis,
        "총회차": total_episode,
        "URL": detail_url
    })

    time.sleep(1)


# 랭킹 수집
ranking_results = []

ranking_url = "https://www.vigloo.com/ko?tab=ranking"
print("\n===== 랭킹 수집 시작 =====")

driver.get(ranking_url)
time.sleep(3)
scroll_down(5)

cards = driver.find_elements(
    By.XPATH,
    "//div[contains(@class,'bg-elevation-100') and .//img]"
)

print("랭킹 카드 개수:", len(cards))

ranking_items = []

for idx, card in enumerate(cards, start=1):
    try:
        text = card.text.strip()
        lines = clean_text_list(text)

        rank = ""
        title = ""
        tags = []

        for line in lines:
            if line.isdigit():
                rank = line
                break

        candidates = [
            line for line in lines
            if line not in [rank, "HOT", "NEW"]
        ]

        if candidates:
            title = candidates[0]
            tags = candidates[1:]

        if title:
            ranking_items.append({
                "rank": rank,
                "title": title,
                "tags": ", ".join(tags),
                "card_index": idx
            })

    except Exception as e:
        print("랭킹 카드 기본정보 수집 오류:", e)

print("랭킹 수집 대상:", len(ranking_items))


for item in ranking_items:
    print(f"랭킹 / {item['rank']} / {item['title']}")

    driver.get(ranking_url)
    time.sleep(2)
    scroll_down(max(1, item["card_index"] // 4))

    cards = driver.find_elements(
        By.XPATH,
        "//div[contains(@class,'bg-elevation-100') and .//img]"
    )

    try:
        target_card = cards[item["card_index"] - 1]

        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", target_card)
        time.sleep(0.5)
        driver.execute_script("arguments[0].click();", target_card)
        time.sleep(3)

    except Exception as e:
        print("랭킹 카드 클릭 실패:", item["title"], e)
        continue

    detail_url = driver.current_url

    body_text = driver.find_element(By.TAG_NAME, "body").text
    lines = clean_text_list(body_text)

    synopsis_candidates = [
        line for line in lines
        if len(line) >= 40
        and "멤버십 구독" not in line
        and "무제한 시청하기" not in line
        and "비슷한 콘텐츠" not in line
        and "에피소드" not in line
    ]

    synopsis = max(synopsis_candidates, key=len) if synopsis_candidates else ""

    detail_genre = ""
    for line in lines:
        if "·" in line and len(line) <= 40:
            detail_genre = line
            break

    total_episode = get_total_episode(body_text)

    ranking_results.append({
        "차트": "랭킹",
        "순번": item["rank"],
        "제목": item["title"],
        "태그/설명": item["tags"],
        "상세_장르": detail_genre,
        "줄거리": synopsis,
        "총회차": total_episode,
        "URL": detail_url
    })

    time.sleep(1)


# 인기 + 랭킹 합치기
popular_df = pd.DataFrame(popular_results)
ranking_df = pd.DataFrame(ranking_results)

df = pd.concat([popular_df, ranking_df], ignore_index=True)

df.to_csv("vigloo_popular_ranking_total.csv", index=False, encoding="utf-8-sig")
df.to_excel("vigloo_popular_ranking_total.xlsx", index=False)

print("\n완료!")
print(df.head())

driver.quit()


===== 인기 수집 시작 =====
인기 카드 개수: 39
인기 수집 대상: 39
인기 / 1 / 국가권력급 상속자가 돌아왔다
인기 / 2 / 천재 아기의 인생 역전
인기 / 3 / 장녀 결혼 일기
인기 / 4 / 비서할래 와이프할래?
인기 / 5 / 타임리프 로맨스: 은밀한 복수
인기 / 6 / 모쏠지옥
인기 / 7 / 오늘 한류스타와 이혼하겠습니다
인기 / 8 / 뚱보 아내는 조폭 마누라
인기 / 9 / 수상한 남편의 이중생활
인기 / 10 / 뽀삐가 재벌남으로 돌아왔다
인기 / 11 / 천재 아기의 인생 역전
인기 / 12 / 이 섬에서만 할 수 있어
인기 / 13 / 엄마가 나를 팔았다
인기 / 14 / 모쏠지옥
인기 / 15 / 내 남편의 여자들
인기 / 16 / 비서와 사랑에 빠질 확률
인기 / 17 / 마지막 베팅 ~금단의 사랑에 빠지다~
인기 / 18 / 수상한 남편의 이중생활
인기 / 19 / Dr.K
인기 / 20 / 절친 아빠를 탐하다
인기 / 21 / 그 여자를 유혹해줘
인기 / 22 / 남편을 유혹해줘요
인기 / 23 / 꿈에서 자유로
인기 / 24 / 당신은 나의 신부
인기 / 25 / 뚱보 아내는 조폭 마누라
인기 / 26 / D-100 : 남편을 리셋합니다
인기 / 27 / 의치한 스캔들
인기 / 28 / 심장이 기억하는 사랑
인기 / 29 / 멍청한 쓰레기 남편은 즉시 폐기!
인기 / 30 / 낮져밤이 오피스
인기 / 31 / 불장난
인기 / 32 / 엘러베이터 S2
인기 / 33 / 완벽한 사모님, CEO를 울리다
인기 / 34 / 벙어리 아내를 구한 신선
인기 / 35 / 닿을 수 있는 그녀
인기 / 36 / 엘러베이터 S1
인기 / 37 / 시혼궁녀
인기 / 38 / 해야만 하는 쉐어하우스 in Japan
인기 / 39 / 여왕벌맘 위에, 히든 재벌맘

===== 랭킹 수집 시작 =====
랭킹 카드 개수: 10
랭킹 수집 대상: 10
랭킹 /  / 천재 아기의 인생 역전
랭킹 /  / 수상한 남편의 이중생활
랭킹 /  

In [16]:
df_merged = (
    df.groupby("제목", as_index=False)
    .agg({
        "차트": lambda x: " + ".join(sorted(set(x))),
        "순번": lambda x: " / ".join(map(str, x)),
        "태그/설명": lambda x: " / ".join(sorted(set(x.dropna()))),
        "상세_장르": "first",
        "줄거리": "first",
        "총회차": "first",
        "URL": "first"
    })
)

df_merged

,제목,차트,순번,태그/설명,상세_장르,줄거리,총회차,URL
0,D-100 : 남편을 리셋합니다,인기,26,타임루프,반전· · 환생· · 사이다,"결혼식 당일, 강예린은 약혼자 남태준과 절친 권유나의 배신으로 목숨을 잃는다. 죽음...",49,https://www.vigloo.com/ko/video/15001590?episo...
1,Dr.K,인기,19,로코,SF· · 로맨스,에너지 고갈로 멸망 위기에 처한 ‘오리진’행성. 최고의 요원 닥터 K는 미지의 에너...,52,https://www.vigloo.com/ko/video/15001750?episo...
2,국가권력급 상속자가 돌아왔다,랭킹 + 인기,1 /,"참교육 / 참교육, 재벌, 반전, 복수, 극적인",재벌· · 반전· · 사이다,세계 10대 기업 GX그룹의 유일한 상속자이자 국방부 장관의 아들인 최산. 그는 여...,63,https://www.vigloo.com/ko/video/15001189?episo...
3,그 여자를 유혹해줘,인기,21,불륜,치정· · 로맨스,"재벌 외동딸 이주하는 평직원 남편 김규진과 결혼해 단란한 삶을 살았지만, 갑작스러운...",49,https://www.vigloo.com/ko/video/15001717?episo...
4,꿈에서 자유로,인기,23,청춘물,판타지· · 로맨스· · 스릴러,평범한 여고생 최정민은 학교폭력 가해자이자 교내 권력자인 하주현의 괴롭힘 속에 지옥...,46,https://www.vigloo.com/ko/video/15001684?episo...
5,남편을 유혹해줘요,인기,22,삼각관계,치정· · 스릴러,미술대학 교수 주원과 갤러리 관장 유진은 10년 차 부부다. 어느 날부터 주원의 행...,43,https://www.vigloo.com/ko/video/15001685?episo...
6,낮져밤이 오피스,인기,30,로코,로맨스· · 코미디,수직사회의 끝판왕. K- 회사. 그중에서도 회사의 먹이사슬 최하위 포지션에 위치한 ...,49,https://www.vigloo.com/ko/video/15001522?episo...
7,내 남편의 여자들,인기,15,불륜,배신· · 스릴러,아끼는 동생의 배신으로 남편과 아이를 모두 잃고 정신병원에 갇힌 여자가 전신성형 후...,54,https://www.vigloo.com/ko/video/15001785?episo...
8,당신은 나의 신부,인기,24,선결혼후연애,로맨스· · 반전,"원계는 친부모와 양부모 모두에게서 외면받으며 외로이 자라왔다. 그러던 중, 거물급 ...",112,https://www.vigloo.com/ko/video/15001652?episo...
9,닿을 수 있는 그녀,랭킹 + 인기,35 /,"인기급상승 3위 / 초고속결혼, 재벌, 로맨틱한",로맨스· · 재벌· · 사이다,강씨 가문의 양녀 강유진은 가족의 지시로 향한 호텔에서 여자 알러지가 있는 성원그룹...,81,https://www.vigloo.com/ko/video/15001453?episo...


In [17]:
from collections import Counter

genres = []

for g in df_merged["상세_장르"]:
    if pd.notna(g):
        genres += [x.strip() for x in g.split("·")]

Counter(genres)

Counter({'': 58,
         '로맨스': 26,
         '사이다': 9,
         '코미디': 7,
         '반전': 6,
         '재벌': 6,
         '배신': 5,
         '판타지': 4,
         '스릴러': 4,
         '드라마': 4,
         '리얼리티': 4,
         '환생': 3,
         '치정': 3,
         '걸크러쉬': 2,
         '다정남': 2,
         'SF': 1,
         '삼각관계': 1,
         '시대극/사극': 1,
         '호러': 1,
         '금지된사랑': 1,
         '연상연하': 1,
         '고자극': 1,
         '운명': 1})

In [18]:
df_merged[df_merged["차트"].str.contains("인기") & df_merged["차트"].str.contains("랭킹")]

,제목,차트,순번,태그/설명,상세_장르,줄거리,총회차,URL
2,국가권력급 상속자가 돌아왔다,랭킹 + 인기,1 /,"참교육 / 참교육, 재벌, 반전, 복수, 극적인",재벌· · 반전· · 사이다,세계 10대 기업 GX그룹의 유일한 상속자이자 국방부 장관의 아들인 최산. 그는 여...,63,https://www.vigloo.com/ko/video/15001189?episo...
9,닿을 수 있는 그녀,랭킹 + 인기,35 /,"인기급상승 3위 / 초고속결혼, 재벌, 로맨틱한",로맨스· · 재벌· · 사이다,강씨 가문의 양녀 강유진은 가족의 지시로 향한 호텔에서 여자 알러지가 있는 성원그룹...,81,https://www.vigloo.com/ko/video/15001453?episo...
10,뚱보 아내는 조폭 마누라,랭킹 + 인기,8 / 25 /,"불륜 / 불륜, 복수, 사이다, 빙의, 정치 / 인기급상승 4위",환생· · 걸크러쉬· · 사이다,대통령 후보 현우의 아내 지애는 남편의 내연녀 채원의 계략으로 죽음의 위기에 처한다...,75,https://www.vigloo.com/ko/video/15001651?episo...
13,모쏠지옥,랭킹 + 인기,6 / 14 /,"고자극 / 고자극, 청춘물, 연상연하, 키스 / 인기급상승 7위",로맨스· · 리얼리티· · 드라마,"친구에게 속아 연애서바이벌 <핫연애존>에 입주한 솔지, 쉴 틈 없이 이어지는 고자극...",53,https://www.vigloo.com/ko/video/15001786?episo...
17,비서할래 와이프할래?,랭킹 + 인기,4 /,"선결혼후연애 / 선결혼후연애, 오피스, 로코, 재벌",로맨스· · 코미디,할머니 수술비가 필요했던 박하은은 1년간 결혼 신분을 빌려주는 계약 결혼을 하기로 ...,80,https://www.vigloo.com/ko/video/15001355?episo...
19,수상한 남편의 이중생활,랭킹 + 인기,9 / 18 /,"로맨틱한 / 로맨틱한, 원나잇, 재벌, 언더커버 / 인기급상승 2위",로맨스· · 재벌· · 반전,"재벌가 후계자 자리를 박차고 나와 택시기사, 정육점, 택배기사로 ‘쓰리 잡’을 뛰며...",72,https://www.vigloo.com/ko/video/15001753?episo...
26,오늘 한류스타와 이혼하겠습니다,랭킹 + 인기,7 /,"이혼 / 이혼, 삼각관계, 사이다",로맨스· · 배신· · 걸크러쉬,대한민국 최고의 한류스타 최현과 비밀 결혼을 한 강혜인. 철저히 숨겨진 결혼 생활 ...,70,https://www.vigloo.com/ko/video/15000826?episo...
30,장녀 결혼 일기,랭킹 + 인기,3 /,"초고속결혼 / 초고속결혼, 재벌, 신데렐라",로맨스· · 재벌· · 사이다,"가난한 토스트 가게 주인 선영은 길가에 쓰러진 은하를 도와준 인연으로, 은하의 오빠...",71,https://www.vigloo.com/ko/video/15000728?episo...
32,천재 아기의 인생 역전,랭킹 + 인기,2 / 11 /,"인기급상승 1위 / 참교육 / 참교육, 비밀신분, 타임루프, 배신",환생· · 반전· · 사이다,친부와 새엄마 때문에 억울한 누명을 쓰고 교도소에서 숨진 도혁. 눈을 떠보니 201...,76,https://www.vigloo.com/ko/video/15001849?episo...
33,타임리프 로맨스: 은밀한 복수,랭킹 + 인기,5 /,"시간여행 / 시간여행, 복수, 불륜, 고자극",로맨스· · 판타지· · 운명,남편 진호와 친구 예린의 불륜을 목격한 하나는 예린에게 살해당하지만 10년 전 과거...,71,https://www.vigloo.com/ko/video/15000140?episo...


In [20]:
df_merged["총회차"].describe()

count     35.000000
mean      59.057143
std       15.210180
min       34.000000
25%       49.000000
50%       55.000000
75%       70.500000
max      112.000000
Name: 총회차, dtype: float64

In [21]:
tags = []

for t in df_merged["태그/설명"]:
    if pd.notna(t):
        tags += t.split(" / ")

Counter(tags).most_common(10)

[('로코', 4),
 ('불륜', 3),
 ('다각관계', 3),
 ('참교육', 2),
 ('선결혼후연애', 2),
 ('엘리트', 2),
 ('신분오해', 2),
 ('타임루프', 1),
 ('참교육, 재벌, 반전, 복수, 극적인', 1),
 ('청춘물', 1)]